In [ ]:
import pandas as pd
import numpy as np
import os
import json
import ast
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from numpy import var, mean, sqrt
from pandas import Series
import itertools
from multiprocessing import Pool
import random
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from scipy.spatial.distance import jensenshannon
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import itertools
from joblib import Parallel, delayed
import warnings
import json
warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:
# Change the current working directory
os.chdir(r"C:\Users\mikke\Documents\GitHub\P3-LDA")

# Set the file paths for the data and ner files
data_path = os.path.join(r"F:\OneDrive - Aalborg Universitet\P3-Privat\articles_df_deep_Cleaned\articles_df_deep_Cleaned.csv")
ner_path = os.path.join(r"F:\OneDrive - Aalborg Universitet\P3-Privat\Results_ner\Results_ner.csv")

# Read the ner file into a pandas DataFrame
df_ner = pd.read_csv(ner_path, encoding="utf-8")

# Read the data file into a pandas DataFrame
df = pd.read_csv(data_path)

In [ ]:
def convert_string_to_dict(string):
    try:
        return ast.literal_eval(string)
    except ValueError:
        return None  # or some error handling

# Ner values are stored as strings in the DataFrame
# Apply this function to each row in the DataFrame to convert the strings to dictionaries
df_ner['ner'] = df_ner['ner'].apply(convert_string_to_dict)


def combine_strings(ner_list): # Function to combine the words in the ner list into a single string
    combine_strings = ""
    for item in ner_list:
        word = item.get('word')
        word = word.replace(" ","_") # Replace spaces with underscores to make sure a mutli-word entity is not split into multiple entities later
        combine_strings = combine_strings + " " + word
    combine_strings = combine_strings.lower()
    return combine_strings

#Apply the function to each row of the "ner" column and store the results in a new column
df_ner['combined_ner'] = df_ner['ner'].apply(combine_strings)

df_ner = df_ner.rename(columns={"id": "ID"}) #not important but the column names need to be the same for the merge to work

df = df.merge(df_ner[["ID","combined_ner"]], on="ID", how="left") # Merge the ner column into the main DataFrame

df["combined_ner"] = df["combined_ner"].fillna("") # Replace NaN values with empty strings if they exist

df = df.drop_duplicates(subset=['ID']) # Drop duplicates if there shoudl be any




df["Date Release"] = pd.to_datetime(df["Date Release"]) # Convert the date column to datetime format to mate comparison possible
df.sort_values(by="Date Release", inplace=True,ascending=False) # Sort the DataFrame by date
df.reset_index(drop=True, inplace=True)   # Reset the index after sorting

In [ ]:
# dictionary of the different True-Stories and their IDs from the database
full_length_true_stories =  {
    "Hockey" : [1708081, 1658478, 1641769],
    "Plusbus" : [1753446, 1721926, 1690348, 1682265, 1670976],
    "HobroEsport" : [49752, 346188, 399813, 534008, 532010, 370353, 83303],
    "SygehusThisted" : [56821, 1100787, 1668177, 1642017, 980452, 540880, 1140334, 1636887, 1439693, 314044],
    "OversvømmelserAalborg" : [350568, 1100787, 1668177, 1642017, 980452, 540880, 1140334, 1636887, 1439693, 314044],
    "Afganistan" : [1622610, 1593106, 1591021, 1591022, 1590007, 1579951, 1577107],
    "Arbejdsløshed" : [1302994, 1261935, 1182802, 626941, 409975],
    "ArbejdsmiljøBjergbo" : [1079557, 1095529, 1095557, 1121380, 1121353, 1166715, 1499766],
    "KlitterHærværk" : [1236290, 1239463, 1246203, 1292567, 1295603, 1336446, 1338446, 1410780, 1418657, 1418557, 1420671, 1420877, 1424710, 1479956, 1480891, 1480905,]              ,
    "EllingOversvømmelser" : [47977, 53014, 167903, 267650, 338414, 294497, 346151, 351437, 412348, 461047, 1190288, 1236379, 1323490, 1556892, 1644002],
    "AmtoftHavn" : [96855, 118623, 462505, 870553, 900344, 906654, 1001537, 1021012, 1199562],
    "Folkeskolelærer" : [278921, 292903, 317037, 405857, 406341, 406089, 409124, 410963, 413506, 414700, 414701, 450289, 451567, 1120076, 1271779, 1280157],
    "VærftSkagen" : [1151898, 551551, 160832, 65872, 62303, 61508],
    "MikkelHansen" : [1460808, 1443989, 1442354, 1442596, 1442576],
    "ShapingNewTomorrow" : [1745579, 1742673, 1677415, 1623517, 1543846, 1543845, 1528669, 1510759, 1487703, 1455707, 1299666, 1267733, 1191265, 1080076, 725030, 681351, 426928],
    "NovoNordisk" : [1740146, 1396440, 626052, 574430, 290241, 238390, 231161, 56714],
    "JamesWebb" : [1699123, 1704172, 1722198, 1693355, 1693343, 732161, 394963],
    "TeaterNordkraft" : [1475734, 1144595, 716735, 583054, 442476, 1574441, 1548308, 1066003],
    "KræmmermarkedGigantium" : [1632252, 1609779, 1597540, 1152080, 718271, 302486, 254634, 730822],
    "HvidvaskDanskeBank" : [148014, 1492728, 1492727, 1159859, 581722, 523460],
    "SpritBilister" : [1688987, 1247012, 1264632, 1397984, 1312268, 832225, 790362, 425603]
}

In [ ]:


def missing_ids(df, full_length_true_stories): # Function to check if any of the IDs in the hand_picked_stories dictionary are missing from the DataFrame
    id_list = []

    for key, values in full_length_true_stories.items():
        id_list.extend(values)
    missing_ids = set(id_list) - set(df["ID"])
    print(f"Number of missing IDs: {len(missing_ids)}")
    print(f"Missing IDs: {list(missing_ids)}")

missing_ids(df, full_length_true_stories)

In [ ]:
sorted_true_stories_full_length: dict[str, np.ndarray] = {} # Dictionary to store the sorted IDs for each True-Story

def sort_cronologically(df, true_stories_full_length): # Function to sort the IDs in each True-Story by date, if they were not already
    for key in true_stories_full_length:
        sorted_ids = sorted(true_stories_full_length[key], key=lambda x: df.loc[df['ID'] == x, 'Date Release'].values[0], reverse=True)
        sorted_dates = df.loc[df['ID'].isin(sorted_ids), ['Date Release', 'ID']]
        sorted_true_stories_full_length[key] = np.array(sorted_ids)
    return sorted_true_stories_full_length

sorted_true_stories_full_length = sort_cronologically(df, full_length_true_stories)



In [ ]:
# Create an empty DataFrame
df_combinations_true = pd.DataFrame(columns=['Key', 'True-Story'])

# Generate 5 length combinations of the IDs in each True-Story, and store them in the DataFrame
def compute_combinations(sorted_true_stories_full_length):
    for key, values in sorted_true_stories_full_length.items():
        combinations = list(itertools.combinations(values, 5))
        df_key = pd.DataFrame({'Key': key, 'Combination': combinations})
        df_combinations = pd.concat([df_combinations, df_key], ignore_index=True)
    return df_combinations

# Display the resulting DataFrame


key_counts = df_combinations_true['Key'].value_counts()
# print(key_counts)





In [ ]:
random_seed = 1 # Set the random seed
random.seed(random_seed) # Set the random seed, for reproducibility of random results

df_combination_random = pd.DataFrame(columns=['combination_random'])

random_seeds = random.sample(range(1, 10000000), len(df_combinations_true)) # Generate a list of random seeds for the random combinations

for _ in range(len(df_combination_random)): # same as for true stories, but for random combinations of same length
    random_rows = df.sample(n=5, random_state=random_seeds[_]) #sample 5 random rows from the dataframe, with random state set to the random seed
    selected_ids = random_rows["ID"].tolist() 
    df_key = pd.DataFrame({'combination_random': [selected_ids]})
    df_combination_random = pd.concat([df_combination_random, df_key], ignore_index=True)

# Sort the IDs in each combination by date
df_combination_random['combination_random'] = df_combination_random['combination_random'].apply(lambda x: sorted(x, key=lambda id: df.loc[df['ID'] == id, 'Date Release'].values[0]))


    



In [ ]:
def cohend(d1: Series, d2: Series, a, b): # Function to calculate the effect size between two samples
    # a and b are only added to make the listing easier to append later


    # calculate the size of samples
    n1, n2 = len(d1), len(d2)

    # calculate the variance of the samples
    s1, s2 = var(d1, ddof=1), var(d2, ddof=1)

    # calculate the pooled standard deviation
    s = sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2))

    # calculate the means of the samples
    u1, u2 = mean(d1), mean(d2)
    
    # print(f"u1: {u1}, u2: {u2}, s1: {s1}, s2: {s2}, n1: {n1}, n2: {n2}")

    d = (u1 - u2) / s

    return (a, b, d, u1, u2, s1, s2, n1, n2)

In [ ]:

# list of all parameters to be tested

min_df1 =   [0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.00006, 0.00007, 0.00008, 0.00009,]
min_df2 =  [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008, 0.0009]
min_df3 = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009]
min_df4 = [0.01, 0.02, 0.03, 0.04, 0.05,0.06, 0.07, 0.08, 0.09]
min_df5 = [0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7, 0.8, 0.9]

combined_min_df = min_df1 + min_df2 + min_df3 + min_df4 + min_df5 # Combine the different min_df values into a single list

max_df1 =   [0.00001, 0.00002, 0.00003, 0.00004, 0.00005, 0.00006, 0.00007, 0.00008, 0.00009,]
max_df2 =  [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008, 0.0009]
max_df3 = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009]
max_df4 = [0.01, 0.02, 0.03, 0.04, 0.05,0.06, 0.07, 0.08, 0.09]
max_df5 = [0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7, 0.8, 0.9]

combined_max_df = max_df1 + max_df2 + max_df3 + max_df4 + max_df5 # Combine the different max_df values into a single list


# Create a list of all combinations of min_df and max_df values, that does not violate the condition that min_df < max_df and min_df != max_df
min_max_df = [pair for pair in itertools.product(combined_min_df, combined_max_df) if pair[0] < pair[1] and pair[0] != pair[1]]


# Create a list of all combinations of ngram values
ngram = [(1,1), (1,2), (1,3), (2,2), (2,3), (3,3)]

# Create a list of all combinations of feature_size values
feature_size = [100, 1000, 10000, 100000, 1000000]

# Create a list of all combinations of a and b values, these are the values for between weight and root weight respectively
a = [0.0,0.1, 0.2, 0.3, 0.4, 0.5,0.6, 0.7, 0.8, 0.9,1.0]
b = [1.0,0.9, 0.8, 0.7, 0.6, 0.5,0.4, 0.3, 0.2, 0.1,0.0]


a_b_pairs = list(zip(a,b)) # Combine the a and b values into a list of tuples

# Calculate the total number of parameter combinations that could be tested
print(len(min_max_df) * len(ngram) * len(feature_size)*len(a_b_pairs))

# Calculate the total number of combinations with out the a and b values
combination_args = [comb for comb in itertools.product(feature_size, ngram, min_max_df)]


In [ ]:
# to speed up combutations, only 5000 combinations of True-Stories and Random-Stories are used

n = 5000 # Number of True-Stories and Random-Stories to use

df_combinations_true_sample = df_combinations_true.sample(n=n, random_state=random_seed)
df_combinations_random_sample = df_combination_random.sample(n=n, random_state=random_seed)

In [ ]:


df_size = df.shape[0] # Get the size of the DataFrame, used later

# function to test the parameters for a given model type, parameter type, parameters, and sample ratio
def parameter_test(model_type, para_or_ner, params, used_df_ratio, save_path):

    def write_failure_to_json(max_features, ngram_range, min_max_df, a, b, file_path): # Function to write the parameters that failed to a json file
        data = {
            'max_features': max_features,
            'ngram_range': ngram_range,
            'min_df': min_max_df[0],
            'max_df': min_max_df[1],
            'a': a,
            'b': b,
            'd': 0,
            "mean_random": 0,
            "mean_true": 0,
            "var_random": 0,
            "var_true": 0,
            "random_size": 0,
            "true_size": 0,
            "random_raw_data": [],
            "true_raw_data": []
        }
        with open(file_path, 'a') as json_file:
            json_file.write(json.dumps(data) + '\n')
            print("EMPTY")       

    try: # Try to run the code, if it fails, write the parameters to the json file, this only happens if the vecorizer fails to fit the data(if no terms are left after pruning)
        max_features, ngram_range, min_max_df = params

        # Define the text to use for the vectorizer
        if para_or_ner == "para":
            text_column = "Paragraph Stripped"
        elif para_or_ner == "ner":
            text_column = "combined_ner"

        # Define the vectorizer type
        if model_type == "tfidf":
            vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_max_df[0], max_df=min_max_df[1])
        elif model_type == "count":
            vectorizer = CountVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_max_df[0], max_df=min_max_df[1])

        subset_size = int(df_size*used_df_ratio) # Calculate the size of the subset of the DataFrame to use
        
        subset_df = df.sample(n=subset_size, random_state=random_seed) # Sample the subset of the DataFrame to fitting

        vectorized_articles = vectorizer.fit(subset_df[text_column]) # Fit the vectorizer to the subset of the DataFrame

        if not vectorizer.vocabulary_: # If the vectorizer has no vocabulary, write the parameters to the json file with 0 values as resutls. This only happened sometimes when the exception was not caught
            write_failure_to_json(max_features, ngram_range, min_max_df, 0, 0, save_path)

    except ValueError as e:
        if str(e) == "After pruning, no terms remain. Try a lower min_df or a higher max_df.":
            write_failure_to_json(max_features, ngram_range, min_max_df, 0, 0, save_path)
            return
        elif str(e) == "max_df corresponds to < documents than min_df":
            write_failure_to_json(max_features, ngram_range, min_max_df, 0, 0, save_path)
            return
        else:
            print("Not accounted for error:", e) # If the error is not accounted for
            raise e 
    
    # Function to compute the cosine similarity between the combinations of IDs
    def compute_cosine_sim(df, combinations):
        data_ = []
        for combination in combinations: 
            article_indexes = df.loc[df["ID"].isin(combination)].index # Get the indexes of the articles in the combination
            
            article_vectors = vectorized_articles[article_indexes] # Get the vectors for the articles in the combination

            if article_vectors.nnz == 0 or np.all(article_vectors.toarray() == article_vectors[0, 0]): # If the vectors are all 0 or all the same, skip the combination, will skrew the results
                continue
            else:
                cosine_sim = cosine_similarity(article_vectors) # Compute the cosine similarity between the articles in the combination
                between_sim = np.diag(cosine_sim, k=1)[:4] # Get the between similarity
                root_sim = cosine_sim[1:, 0] # Get the root similarity

                combined_list = np.concatenate((between_sim, root_sim)) # Combine the between and root similarity into a single list
                data_.append(combined_list) # Append the list to the data list

        # a1= between similarity between article 1 and 2
        # a2= between similarity between article 2 and 3
        # a3= between similarity between article 3 and 4
        # a4= between similarity between article 4 and 5
        # b1= root similarity article 2 
        # b2= root similarity article 3
        # b3= root similarity article 4
        # b4= root similarity article 5
        df_sim_for_all_combinations = pd.DataFrame(data_, columns=['a1', 'a2', 'a3', 'a4', 'b1', 'b2', 'b3', 'b4'])
        return df_sim_for_all_combinations # Return the DataFrame with the cosine similarity for each combination

    # Use the function to compute the cosine similarity for the True-Stories and Random-Stories
    df_trues_sim = compute_cosine_sim(df, df_combinations_true_sample['Combination'].values)
    df_random_sim = compute_cosine_sim(df, df_combinations_random_sample['combination_random'].values)

    # Convert the DataFrames to numpy arrays for faster computation
    true_array = df_trues_sim.to_numpy()
    random_array = df_random_sim.to_numpy()

    def process_a_b_pair(a, b, max_features, ngram_range, min_df, max_df):
        # Convert to numpy for faster computation

        # Make copies of the arrays to avoid changing the original arrays
        sim_array_copy = true_array.copy()
        random_array_copy = random_array.copy()

        # Scale columns a= between weight, b= root weight
        sim_array_copy[:, :4] *= a
        sim_array_copy[:, -4:] *= b
        random_array_copy[:, :4] *= a
        random_array_copy[:, -4:] *= b
        
        # Compute the sum of each row
        true_combined_sum = sim_array_copy.sum(axis=1)
        random_sum_sum = random_array_copy.sum(axis=1)

        # Normalize
        max_value = 4 # The maximum value of the sum of all the cosine similarity, mentioned in the project, min value is 0
        true_combined_sum /= max_value
        random_sum_sum /= max_value

        # Compute the effect size(Evaluation score in the project)
        results = cohend(random_sum_sum, true_combined_sum, a, b)
        
        (a, b, d, u1, u2, s1, s2, n1, n2) = results
        
        # Write the results to the json file
        data = {
        'max_features': max_features,
        'ngram_range': ngram_range,
        'min_df': min_df,
        'max_df': max_df,
        'a': a,
        'b': b,
        'd': d,
        "mean_random": u1,
        "mean_true": u2,
        "var_random": s1,
        "var_true": s2,
        "random_size": n1,
        "true_size": n2,
        "random_raw_data": random_sum_sum.tolist(),
        "true_raw_data": true_combined_sum.tolist()
          }
        with open(save_path, 'a') as json_file:
            json_file.write(json.dumps(data) + '\n')



    # Check if the DataFrames are empty, if they are, write the parameters to the json file with 0 values as results
    # This only happened if almost of the data all have vectors of all 0. 
    # This was done to make sure the cohens d function did not fail, as it requires at least 3 values in each sample, due to the degrees of freedom was 2, and the standard deviation 
    # only result with 5000 sample still in the results were compared later, so this doesnt skrew the results in the project
    # This is only for error handling.
    if len(df_trues_sim) < 2 or len(df_random_sim) < 2:
        write_failure_to_json(max_features, ngram_range, min_max_df, 0, 0, save_path)
        print("Sizes are less than 3")
        return

    
    # create list of arguments to be passed passed to the process_a_b_pair function which is the last function to be called and create the output
    args = [(a, b, max_features, ngram_range, min_max_df[0], min_max_df[1] ) for a, b in a_b_pairs]
    
    # call the process_a_b_pair function with the arguments
    for i in args:
        process_a_b_pair(*i)    

    print("DONE") # Print done when the function is done, to keep track of the progress



def worker(param_entry, file_path): # Function to call the parameter_test function with the correct parameters
    model_type, para_or_ner,params, df_sample_ratio = param_entry # Unpack the parameters
    parameter_test(model_type, para_or_ner,params, df_sample_ratio,file_path) # Call the parameter_test function with the parameters




# process_model_parameter_search is the main function to be called, it takes the model type, 
# parameter type, parameters, file path and sample ratio as arguments
    
def process_model_parameter_search(model_type, para_or_ner, params, file_path, df_sample_ratio): # Function to call the worker function with the correct parameters
    combined_params_list = []
    for arg_combination in params: # Create a list of all the parameters to be tested

        combined_params_list.append([model_type, para_or_ner, arg_combination, df_sample_ratio]) # Append the parameters to the list

    counter = 0 
    for param in combined_params_list: # Call the worker function with the parameters
        counter += 1
        print(f"Processing {counter} of {len(combined_params_list)}") # Print the progress
        worker(param,file_path) # Call the worker function with the parameters


In [ ]:
# random_seed = 1 # Tf-idf-ner
# random_seed = 2 # Count-ner
# random_seed = 3 # Tf-idf-paragraph
# random_seed = 4 # Count-paragraph

random.seed(random_seed)

arg_sample = random.sample(combination_args, 500) # Sample 500 combinations of parameters to test, differnt for each model


# used for the default parameters runs
arg_sample_default = [
    ("tfidf", "ner", [(None, (1, 1), (1, 1.0))], "TF_IDF_NER_default_result_df.json" , 0.3),
    ("count", "ner", [(None, (1, 1), (1, 1.0))], "COUNT_NER_default_result_df.json", 0.3),
    ("tfidf", "para", [(None, (1, 1), (1, 1.0))], "TF_IDF_PARA_default_result_df.json", 0.3),
    ("count", "para", [(None, (1, 1), (1, 1.0))], "COUNT_PARA_default_result_df.json", 0.3)
]

for arg in arg_sample_default:
    process_model_parameter_search(*arg)


# all runs done is seen below
#process_model_parameter_search("tfidf", "ner", arg_sample, "TF_IDF_NER_SEARH_result_df.json", 0.3) # Mikkel
#process_model_parameter_search("count", "ner", arg_sample, "COUNT_NER_SEARH_result_df.json", 0.3) # Gonde
#process_model_parameter_search("tfidf", "para", arg_sample, "TF_IDF_PARA_SEARH_result_df.json", 0.3) # Gvidas
#process_model_parameter_search("count", "para", arg_sample, "COUNT_PARA_SEARH_result_df.json", 0.3) # Dani